# CardioIA Fase 2 — Extração de sintomas

**Guilherme Yamada Dantas · RM 568506**

Este notebook lê os dez relatos e o mapa de conhecimento, encontra expressões e apresenta hipóteses didáticas justificadas. Não realiza diagnóstico nem triagem clínica. Todos os relatos são fictícios.

## 1. Ler os arquivos

O TXT tem uma frase por linha. Cada regra exige os dois sintomas da linha do CSV. As associações são simplificações educacionais, não critérios diagnósticos validados.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'assets/dados').is_dir(), 'Abra o notebook dentro do repositório.'
sys.path.insert(0, str(ROOT))
from src.extracao import carregar_mapa, analisar_frase, extrair_arquivo
mapa = carregar_mapa()
display(pd.DataFrame(mapa))
frases = (ROOT / "assets/dados/frases_sintomas.txt").read_text(encoding="utf-8").splitlines()
assert len(frases) == 10
display(pd.DataFrame({"relato": range(1, 11), "frase": frases}))

,Sintoma 1,Sintoma 2,Doença Associada
0,dor no peito,suor frio,Infarto agudo do miocárdio (associação didática)
1,dor no peito,falta de ar,Infarto agudo do miocárdio (associação didática)
2,falta de ar,inchaço nas pernas,Insuficiência cardíaca (associação didática)
3,cansaço,inchaço nas pernas,Insuficiência cardíaca (associação didática)


,relato,frase
0,1,Desde esta manhã sinto dor no peito e suor fri...
1,2,Há dois dias tenho falta de ar e inchaço nas p...
2,3,Desde ontem percebo cansaço e inchaço nas pern...
3,4,Há meia hora comecei a sentir dor no peito e f...
4,5,"Desde ontem tenho dor no peito, falta de ar e ..."
5,6,Desde a corrida desta manhã sinto cansaço leve...
6,7,"Desde ontem tenho coceira no braço, que incomo..."
7,8,Minha mãe teve dor no peito e suor frio ontem;...
8,9,"Desde a madrugada tenho falta de ar e cansaço,..."
9,10,"Há uma hora sinto DOR NO PEITO e SUOR FRIO, e ..."


## 2. Extrair sintomas e explicar hipóteses

Normalizamos caixa e acentos sem modificar o relato exibido. Correspondências exigem limites de palavra. O extrator usa regras simples de negação e sujeito; linguagem livre ainda pode ultrapassar esses padrões.

In [2]:
resultados = extrair_arquivo()
tabela = pd.DataFrame([{
    "relato": i, "frase": r["frase"], "afirmados": ", ".join(r["sintomas"]),
    "negados": ", ".join(r["negados"]),
    "hipóteses didáticas": "; ".join(h["doenca"] for h in r["hipoteses"]),
    "status": r["status"]
} for i,r in enumerate(resultados, 1)])
display(tabela)
print("Dez relatos processados:", len(resultados))

,relato,frase,afirmados,negados,hipóteses didáticas,status
0,1,Desde esta manhã sinto dor no peito e suor fri...,"dor no peito, suor frio",,Infarto agudo do miocárdio (associação didática),hipotese_encontrada
1,2,Há dois dias tenho falta de ar e inchaço nas p...,"falta de ar, inchaço nas pernas",,Insuficiência cardíaca (associação didática),hipotese_encontrada
2,3,Desde ontem percebo cansaço e inchaço nas pern...,"inchaço nas pernas, cansaço",,Insuficiência cardíaca (associação didática),hipotese_encontrada
3,4,Há meia hora comecei a sentir dor no peito e f...,"dor no peito, falta de ar",,Infarto agudo do miocárdio (associação didática),hipotese_encontrada
4,5,"Desde ontem tenho dor no peito, falta de ar e ...","dor no peito, falta de ar, inchaço nas pernas",,Infarto agudo do miocárdio (associação didátic...,hipotese_encontrada
5,6,Desde a corrida desta manhã sinto cansaço leve...,cansaço,"dor no peito, falta de ar",,sem_correspondencia
6,7,"Desde ontem tenho coceira no braço, que incomo...",,"dor no peito, suor frio",,sem_correspondencia
7,8,Minha mãe teve dor no peito e suor frio ontem;...,cansaço,,,sem_correspondencia
8,9,"Desde a madrugada tenho falta de ar e cansaço,...","falta de ar, cansaço",inchaço nas pernas,,sem_correspondencia
9,10,"Há uma hora sinto DOR NO PEITO e SUOR FRIO, e ...","dor no peito, suor frio",,Infarto agudo do miocárdio (associação didática),hipotese_encontrada


Dez relatos processados: 10


## 3. Inspecionar a justificativa

A saída não escolhe uma hipótese arbitrariamente quando mais de uma regra coincide. Mostra as expressões responsáveis por cada associação.

In [3]:
display(resultados[4])

{'frase': 'Desde ontem tenho dor no peito, falta de ar e inchaço nas pernas, e precisei faltar ao trabalho.',
 'sintomas': ['dor no peito', 'falta de ar', 'inchaço nas pernas'],
 'negados': [],
 'hipoteses': [{'doenca': 'Infarto agudo do miocárdio (associação didática)',
   'sintomas': ['dor no peito', 'falta de ar']},
  {'doenca': 'Insuficiência cardíaca (associação didática)',
   'sintomas': ['falta de ar', 'inchaço nas pernas']}],
 'status': 'hipotese_encontrada'}

## 4. Casos de controle

Testamos negação, antecedente familiar, ausência de vocabulário e contradição. Quando o mesmo sintoma é afirmado e negado, as hipóteses são suspensas; não inventamos uma sequência temporal.

In [4]:
controles = [
    "Não sinto dor no peito nem suor frio.",
    "Minha mãe teve dor no peito e suor frio; eu estou bem.",
    "Tenho coceira no braço.",
    "Sem dúvida sinto dor no peito e suor frio.",
    "Sinto dor no peito e suor frio. Não sinto dor no peito.",
]
display(pd.DataFrame([analisar_frase(x) for x in controles]))
assert analisar_frase(controles[0])["hipoteses"] == []
assert analisar_frase(controles[-1])["status"] == "contexto_ambiguo"

,frase,sintomas,negados,hipoteses,status
0,Não sinto dor no peito nem suor frio.,[],"[dor no peito, suor frio]",[],sem_correspondencia
1,Minha mãe teve dor no peito e suor frio; eu es...,[],[],[],sem_correspondencia
2,Tenho coceira no braço.,[],[],[],sem_correspondencia
3,Sem dúvida sinto dor no peito e suor frio.,"[dor no peito, suor frio]",[],[{'doenca': 'Infarto agudo do miocárdio (assoc...,hipotese_encontrada
4,Sinto dor no peito e suor frio. Não sinto dor ...,"[dor no peito, suor frio]",[dor no peito],[],contexto_ambiguo


## 5. O que esta implementação não resolve

O mapa contém quatro associações. Sinônimos não cadastrados, citações, temporalidade e escopo complexo de negação podem falhar. Sem correspondência não significa ausência de doença. Sintomas são compartilhados por diversas condições; uma regra acionada não confirma nenhuma delas.

**Referências:** [NHLBI — sintomas de infarto](https://www.nhlbi.nih.gov/health/heart-attack/symptoms), [NHLBI — sintomas de insuficiência cardíaca](https://www.nhlbi.nih.gov/health/heart-failure/symptoms). As fontes apoiam o vocabulário, não validam nossas conjunções. Consulte `document/dados-e-ontologia.md` para proveniência e limitações.